In [4]:
import pandas as pd
import requests
import geopandas as gpd
from rasterstats import zonal_stats
import os, glob
import rasterio
import numpy as np
from rasterio.features import rasterize
from datetime import datetime
from dateutil.relativedelta import relativedelta
import re

raster_folder = "./data"

In [5]:
def get_key_from_environment(file_path: str, key: str) -> str | None:
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Regex to match key: 'value' or key: "value"
    pattern = rf'{key}\s*:\s*[\'"]([^\'"]+)[\'"]'
    match = re.search(pattern, content)

    return match.group(1) if match else None


# Example usage
file_path = "../src/environments/environment.ts"
api_key = get_key_from_environment(file_path, "apiToken")

In [7]:
def get_data(scale):
    header = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }


    start_date = datetime(2020, 9, 1)   # Sept 2024
    end_date = datetime(2025, 8, 1)     # Aug 2025

    # Loop over months
    date = start_date
    while date <= end_date:
        date_str = date.strftime("%Y-%m")
        url = f"https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale{scale:03d}&date={date_str}"
        res = requests.get(url, headers=header)
        
        if res.status_code == 200:
            file = f"./data/spi{scale:03d}_{date_str}.tif"
            with open(file, "wb") as f:
                f.write(res.content)
        
        # Move to next month
        date += relativedelta(months=1)

In [ ]:
def get_averages(scale, division, id_col):
    shp_path = f"../public/shapefiles/{division}.shp"

    # Load shapefile
    gdf = gpd.read_file(shp_path)
    gdf = gdf.reset_index(drop=True)

    # --- Handle duplicates depending on division type ---
    if division in ["county", "climate"]:
        # Dissolve by id_col (merging same-name geometries into one)
        gdf = gdf.dissolve(by=id_col, as_index=False)
    else:
        # Make IDs unique by adding -1, -2, etc. for duplicates
        counts = {}
        unique_names = []
        for name in gdf[id_col]:
            if name in counts:
                counts[name] += 1
                unique_names.append(f"{name}-{counts[name]}")
            else:
                counts[name] = 0
                unique_names.append(name)
        gdf[id_col] = unique_names

    # Collect rasters
    tifs = sorted(glob.glob(os.path.join(raster_folder, f"spi{scale:03d}_*.tif")))
    if not tifs:
        raise FileNotFoundError("No rasters found")

    # Get raster metadata
    with rasterio.open(tifs[0]) as src:
        raster_crs = src.crs
        transform = src.transform
        shape = (src.height, src.width)

    if gdf.crs != raster_crs:
        gdf = gdf.to_crs(raster_crs)

    # Rasterize polygons once
    shapes = [(geom, idx) for idx, geom in enumerate(gdf.geometry)]
    mask = rasterize(shapes, out_shape=shape, transform=transform, fill=-1, dtype="int32")

    records = []

    for tif in tifs:
        date = os.path.basename(tif).split("_")[1].replace(".tif", "")
        with rasterio.open(tif) as src:
            arr = src.read(1, masked=True)

        for idx, div in enumerate(gdf[id_col]):
            poly_mask = mask == idx
            vals = arr[poly_mask]

            # Drop masked values cleanly
            if np.ma.is_masked(vals):
                vals = vals.compressed()
            mean_val = np.nan if vals.size == 0 else np.nanmean(vals)
            records.append({"division": div, "date": date, "mean_spi": mean_val})

    df = pd.DataFrame(records)

    df = (
        df.groupby(["division", "date"])["mean_spi"]
        .mean()
        .reset_index()
    )

    df = df.pivot(index="division", columns="date", values="mean_spi")
    df = df.reindex(sorted(df.columns), axis=1)

    out_csv = f"../public/{division}_spi{scale}.csv"
    df.to_csv(out_csv)
    print(f"Saved {out_csv}")


In [10]:
division = "county"
shp_path = f"../public/shapefiles/{division}.shp"
id_col = "county"
# Load shapefile
gdf = gpd.read_file(shp_path)
gdf = gdf.reset_index(drop=True)

# --- Handle duplicates depending on division type ---
if division in ["county", "climate"]:
    gdf = gdf.dissolve(by=id_col, as_index=False)
else:
    # Make IDs unique by adding -1, -2, etc. for duplicates
    counts = {}
    unique_names = []
    for name in gdf[id_col]:
        if name in counts:
            counts[name] += 1
            unique_names.append(f"{name}-{counts[name]}")
        else:
            counts[name] = 0
            unique_names.append(name)
    gdf[id_col] = unique_names

# Collect rasters
tifs = sorted(glob.glob(os.path.join(raster_folder, f"*.tif")))
if not tifs:
    raise FileNotFoundError("No rasters found")


In [20]:
scale = 1
# Get raster metadata
with rasterio.open(tifs[0]) as src:
    raster_crs = src.crs
    transform = src.transform
    shape = (src.height, src.width)

if gdf.crs != raster_crs:
    gdf = gdf.to_crs(raster_crs)

# Rasterize polygons once
shapes = [(geom, idx) for idx, geom in enumerate(gdf.geometry)]
mask = rasterize(shapes, out_shape=shape, transform=transform, fill=-1, dtype="int32")

records = []

for tif in tifs:
    date = os.path.basename(tif).split("_")[1].replace(".tif", "")
    with rasterio.open(tif) as src:
        arr = src.read(1, masked=True)

    for idx, div in enumerate(gdf[id_col]):
        poly_mask = mask == idx
        vals = arr[poly_mask]

        # Drop masked values cleanly
        if np.ma.is_masked(vals):
            vals = vals.compressed()
        mean_val = np.nan if vals.size == 0 else np.nanmean(vals)
        records.append({"division": div, "date": date, "mean_spi": mean_val})

df = pd.DataFrame(records)

df = (
    df.groupby(["division", "date"])["mean_spi"]
    .mean()
    .reset_index()
)

df = df.pivot(index="division", columns="date", values="mean_spi")
df = df.reindex(sorted(df.columns), axis=1)

out_csv = f"../public/{division}_spi{scale}.csv"
# df.to_csv(out_csv)
print(f"Saved {out_csv}")


Saved ../public/county_spi1.csv


In [26]:
dataset = 'rainfall'
scale= None
out_path = f"{raster_folder}/" \
   f"{dataset if dataset != 'drought' else ''}" \
   f"{f'spi{scale:03d}' if scale else ''}" \
   f"_{'params'}.tif"
out_path


'./data/rainfall_params.tif'

In [13]:
df

date,2020-09,2020-10,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,...,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08
division,,,,,,,,,,,,,,,,,,,,,
Hawaiʻi,0.335642,-0.454993,0.615786,-0.216135,1.030925,-0.035914,0.999493,0.310460,0.146973,-0.137363,...,0.614793,-1.766645,0.515741,-1.216250,-0.105854,-0.209330,-0.127303,0.233200,-0.522715,-0.759703
Honolulu,-1.649179,0.106519,-0.182066,-0.599713,1.099467,0.271073,1.972930,-0.921655,-0.813912,-0.926087,...,-0.569648,-1.470771,1.172348,-1.042291,-0.544770,1.031537,0.377160,-0.514973,-0.366765,-1.782338
Kauaʻi,-0.557576,-0.262799,0.006769,-0.645065,0.171160,0.636084,1.926204,-0.680092,-0.201422,-0.213343,...,-0.915459,-1.165575,0.180432,-0.875042,-0.755270,1.323352,0.633160,0.085115,-0.371396,-1.376920
Maui,-0.569951,0.183397,-0.527222,-1.054808,0.638348,-0.118844,1.414761,-0.085574,-0.900565,-1.369345,...,-0.757176,-1.697208,1.094768,-1.350742,-1.001252,0.296227,-0.686878,-0.690920,-0.732647,-0.975898


In [14]:
scales= [1,6,12]
for scale in scales:
    get_data(scale)
    get_averages(scale, "county", "county")
    get_averages(scale, "moku", "moku")
    get_averages(scale, "ahupuaa", "ahupuaa")
    get_averages(scale, "climate", "name")
    get_averages(scale, "watershed", "name")

    folder = "./data"

    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        if os.path.isfile(file_path):  # ensures it's a file
            os.remove(file_path)

KeyboardInterrupt: 

In [81]:
#statewide
shapefile = "../public/shapefiles/Coastline.shp"
raster_folder = "./data"

# Load shapefile and dissolve to one feature (statewide)
gdf = gpd.read_file(shapefile)
gdf_statewide = gdf.dissolve()  # merges all polygons into one

records = []

for tif in sorted(glob.glob(os.path.join(raster_folder, f"spi{scale:03d}*.tif"))):
    # Extract date from filename (spi001_2024-09.tif → 2024-09)
    date = os.path.basename(tif).split("_")[1].replace(".tif", "")
    
    # Compute zonal stats for the dissolved geometry
    stats = zonal_stats(
        vectors=gdf_statewide,
        raster=tif,
        stats=["mean"],
        geojson_out=False,
        nodata=-9999
    )
    
    records.append({
        "date": date,
        "value": stats[0]["mean"]
    })


# Pivot to wide format: one row (statewide), months as columns
df = pd.DataFrame(records).set_index("date").T
df.insert(0, "state", "statewide")  # add proper state column
df.to_csv(f"../public/statewide_spi{scale}.csv", index=False)

print(df)

date       state   2020-09   2020-10   2020-11  2020-12  2021-01   2021-02  \
value  statewide  0.561331  0.479813  0.606564  0.46056  0.26788  0.221295   

date    2021-03  2021-04   2021-05  ...   2024-11   2024-12   2025-01  \
value  0.372087  0.35898  0.294948  ...  0.024834 -0.161592 -0.150209   

date    2025-02   2025-03   2025-04   2025-05   2025-06  2025-07   2025-08  
value -0.269035 -0.095722 -0.171698 -0.439413 -0.401593 -0.40669 -0.875027  

[1 rows x 61 columns]


In [27]:
folder = "./data"

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    if os.path.isfile(file_path):  # ensures it's a file
        os.remove(file_path)